# NB1: Baseline

Our question: Does the improvement in generalisation performance of neural networks when using dropout regularisation hold with increasing training dataset sizes? 

Theory Claim: For smaller trained modules, dropout reduces overfitting, increasing overall generalisation performance.

Why choose 10%? Is this reasonable base to show overfitting?

Our Baseline: Overffited dataset with increasingly applied levels of dropout. 

Run with over 3 random seeds.

Fashion-MNIST - why? 

Need to code: function to train fashion-MNIST and then apply with dropout at difference percentages. 

Might need to pip install torch and pip install torchvision

In [7]:
#libray imports
import pathlib
import numpy as np
import matplotlib.pyplot as plt
import torch, torch.nn as nn
from torchvision import datasets, transforms

device = ('cuda' if torch.cuda.is_available() else
          'mps' if torch.backends.mps.is_available() else 'cpu')

transform = transforms.Compose([transforms.ToTensor()])
train_dataset = datasets.FashionMNIST('./data', train=True, download=True, transform=transform)
test_dataset = datasets.FashionMNIST('./data', train=False, download=True, transform=transform)
print(f'FashionMNIST: {len(train_dataset)} train images, {len(test_dataset)} test images')

FashionMNIST: 60000 train images, 10000 test images


In [8]:
#Settings
HIDDEN_UNITS = 256
TRAIN_PERCENT = 0.1
N_TRAIN = int(len(train_dataset)* TRAIN_PERCENT)
EPOCHS = 5 #We need to work out if we want all epochs no batches or all batch no epoch or a mix (mini-batch)
BATCH = 128
LEARNING_RATE = 0.001
DROPOUT_RATES = [0.0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8]
SEED = [16, 31, 9, 92, 2]

In [9]:
#Loading the dataset into device tensors: 10% of the training set will be used for training

def to_tensor(dataset, index=None):
    images, labels = (dataset.data, dataset.targets) if index is None else (dataset.data[index], dataset.targets[index])
    X = images.float().div(255).view(-1,784)
    return X.to(device), labels.to(device)

#generator = torch.Generator().manual_seed(16)
def random_sample(train_dataset, test_dataset):
    order = torch.randperm(len(train_dataset))[:N_TRAIN]
    x_train, y_train = to_tensor(train_dataset, order)
    x_test, y_test = to_tensor(test_dataset)
    print(f"device {device} | train {tuple(x_train.shape)} | test {tuple(x_test.shape)}")
    return x_train, x_test, y_train, y_test

In [10]:
#Making and training the model

#Why did we choose 2 hidden layers? we could change this ..., standardise this ... or explain why 2 is the best option.
def make_model(dropout_rate):
    return torch.nn.Sequential(
        torch.nn.Linear(784, HIDDEN_UNITS), torch.nn.ReLU(), torch.nn.Dropout(dropout_rate),
        torch.nn.Linear(HIDDEN_UNITS, HIDDEN_UNITS), torch.nn.ReLU(), torch.nn.Dropout(dropout_rate),
        torch.nn.Linear(HIDDEN_UNITS, 10) #There are only 10 types of clothing class in fashionMNIST.
    ).to(device)

@torch.no_grad()
def accuracy(model, X, y):
    model.eval()
    return(model(X).argmax(1) == y).float().mean().item()

def train(model, x_train, y_train, x_test, y_test, seed, verbose=True):
    random_generator = torch.Generator(device="cpu").manual_seed(seed)
    adam_opt = torch.optim.SGD(model.parameters(), lr = LEARNING_RATE)
    loss_function = nn.CrossEntropyLoss()
    results = {"train": [], "test": []}
    for ep in range(EPOCHS):
        model.train()
        shuffle = torch.randperm(len(x_train), generator=random_generator).to(device)
        for i in range(0, len(x_train), BATCH):
            index = shuffle[i:i + BATCH]
            adam_opt.zero_grad()
            loss = loss_function(model(x_train[index]), y_train[index])
            loss.backward()
            adam_opt.step()
        results["train"].append(accuracy(model, x_train, y_train))
        results["test"].append(accuracy(model, x_test, y_test))
    if verbose and (ep + 1) % 10  ==0:
        print(f"epoch {ep} | train {results['train'][-1]:.4f} | test {results['test'][-1]:.4f}")
    return results

In [11]:
#Run with varying dropout
def dropout_experiment(seed):
    models = {}
    results = {}
    x_train, x_test, y_train, y_test = random_sample(train_dataset, test_dataset)

    for r in DROPOUT_RATES:
        models[r] = make_model(r)
        results[r] = train(models[r], x_train, y_train, x_test, y_test, seed)

    print(f"{'Dropout Rate'} {'Training Accuracy'} {'Test Accuracy'} {'Difference'}")
    for r in DROPOUT_RATES:
        train_data = results[r]["train"][-1]
        test_data = results[r]["test"][-1]
        print(f"{r:12.2f} {train_data:17.4f} {test_data:13.4f} {train_data-test_data:10.4f}")

In [12]:
def main():
    for i in SEED:
        print(f"\nSeed: {i}")
        dropout_experiment(i)

main()


Seed: 16
device mps | train (6000, 784) | test (10000, 784)
Dropout Rate Training Accuracy Test Accuracy Difference
        0.00            0.1703        0.1766    -0.0063
        0.10            0.1945        0.1813     0.0132
        0.20            0.2117        0.2191    -0.0074
        0.30            0.2127        0.2176    -0.0049
        0.40            0.1575        0.1601    -0.0026
        0.50            0.1167        0.1125     0.0042
        0.60            0.1827        0.1882    -0.0055
        0.70            0.1257        0.1292    -0.0035
        0.80            0.1365        0.1434    -0.0069

Seed: 31
device mps | train (6000, 784) | test (10000, 784)
Dropout Rate Training Accuracy Test Accuracy Difference
        0.00            0.1722        0.1712     0.0010
        0.10            0.2002        0.1921     0.0081
        0.20            0.1667        0.1644     0.0023
        0.30            0.1982        0.1916     0.0066
        0.40            0.1717        

VARIABILITY ANALYSIS
